# FlowEdit Dir-Mag Five-Sample CLIP/DINO Test

This notebook tests a within-step direction-magnitude hybrid for the CLIP-DINO balance.

Default experiment set, same estimated NFE = 18:

1. `FiveSample_OriginalFlowEdit_Euler18_Tar75`: Euler baseline, `T_steps=24`, `n_max=18`.
2. `FiveSample_BridgeInterp_OldBest_L100_G200`: previous BridgeInterp best, `T_steps=12`, `n_max=9`, `lambda=1.0`, `gamma=2.0`, all-stage correction.
3. `FiveSample_BridgeDirMag_Balanced_T040_W070_M020_R110`: new Dir-Mag candidate, late-stage direction correction plus small clamped magnitude recovery.

Five fresh repository samples are used: `bikes`, `bus`, `cake_red_blueberries`, `flowers`, and `pizza_board`. The previous three stress cases are not used.

Dir-Mag update:

$$G_t = \Delta_{FE}(z_t,t), \quad z_m = z_t + \frac{1}{2}\Delta t G_t, \quad G_m = \Delta_{FE}(z_m,t_m)$$

Direction is rotated toward the bridge midpoint field, while magnitude is only lightly recovered:

$$\hat u_t = \operatorname{normalize}((1-\alpha)u_t + \alpha u_m)$$

$$\hat G_t = \operatorname{clamp\_recover}(\|G_t\|, \|wG_m\|)\hat u_t$$


In [ ]:
# 1) Configuration
# Do not hard-code real tokens into git. Paste one only at runtime if SD3 access requires it.
import os
from getpass import getpass

REPO_URL = "https://github.com/Jiaqi-Ye/FlowEdit.git"
BRANCH = "codex/clip-dino-dir-mag-five-sample"
WORKDIR = "/content/FlowEdit"

EXP_YAML = "SD3_clip_dino_dir_mag_five_sample.yaml"
DATASET_YAML = "edits_clip_dino_five_sample.yaml"
TAG = "dir_mag_five_sample"
PIPELINE_LOAD_MODE = "to_device"  # A100 should fit SD3-medium directly and is faster than CPU offload.

# Optional references. Keep these off for the first half-hour smoke test.
RUN_DIRECTIONAL_REFERENCE = False
RUN_CLIP_RECOVERY_CANDIDATE = False

METRICS_DIR = "outputs/metrics"
QUALITY_PER_SAMPLE_CSV = f"{METRICS_DIR}/{TAG}_clip_dino_per_sample.csv"
QUALITY_SUMMARY_CSV = f"{METRICS_DIR}/{TAG}_clip_dino_summary.csv"
ARTIFACT_PER_SAMPLE_CSV = f"{METRICS_DIR}/{TAG}_artifact_per_sample.csv"
ARTIFACT_SUMMARY_CSV = f"{METRICS_DIR}/{TAG}_artifact_summary.csv"
COMPARISON_CSV = f"{METRICS_DIR}/{TAG}_baseline_delta_summary.csv"
PER_SAMPLE_COMPARISON_CSV = f"{METRICS_DIR}/{TAG}_per_sample_delta.csv"

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if not HF_TOKEN:
    HF_TOKEN = getpass("Paste Hugging Face token, or press Enter if already logged in/cached: ").strip()
if HF_TOKEN:
    os.environ["HF_TOKEN"] = HF_TOKEN


In [ ]:
# 2) Clone repository, install dependencies, and switch to workspace
import os
import subprocess
from pathlib import Path

if not Path(WORKDIR).exists():
    subprocess.run(["git", "clone", REPO_URL, WORKDIR], check=True)

os.chdir(WORKDIR)
subprocess.run(["git", "fetch", "origin"], check=False)
checkout = subprocess.run(["git", "checkout", BRANCH], text=True, capture_output=True)
if checkout.returncode != 0:
    print(checkout.stdout)
    print(checkout.stderr)
    raise RuntimeError(
        f"Could not checkout branch {BRANCH}. Push the branch first or change BRANCH in Cell 1."
    )
subprocess.run(["git", "pull", "--ff-only", "origin", BRANCH], check=False)
subprocess.run(["git", "status", "--short", "--branch"], check=True)

# Keep Colab's binary stack mostly intact. These packages are the ones this repo needs.
subprocess.run([
    "python", "-m", "pip", "install", "-q", "--upgrade",
    "diffusers>=0.31.0", "transformers>=4.44.0", "accelerate>=0.33.0",
    "safetensors", "sentencepiece", "einops", "pyyaml", "huggingface_hub",
    "plotly==5.24.1",
], check=True)


In [ ]:
# 3) Login and inspect GPU
import subprocess
from pathlib import Path

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face.")
else:
    print("No HF token supplied. This works only if the model is already cached or public access is available.")

subprocess.run(["nvidia-smi"], check=False)


In [ ]:
# 4) Inspect experiment and dataset configuration
from pathlib import Path
import pandas as pd
import yaml

required_files = [
    EXP_YAML,
    DATASET_YAML,
    "run_script.py",
    "FlowEdit_utils.py",
    "evaluate_clip_dino.py",
    "evaluate_artifact_proxy.py",
]
missing = [path for path in required_files if not Path(path).exists()]
if missing:
    raise FileNotFoundError(f"Missing required files: {missing}")

with open(EXP_YAML, "r", encoding="utf-8") as f:
    exp = yaml.safe_load(f)
with open(DATASET_YAML, "r", encoding="utf-8") as f:
    dataset = yaml.safe_load(f)

if RUN_DIRECTIONAL_REFERENCE:
    directional = dict(exp[-1])
    directional.update({
        "exp_name": "FiveSample_BridgeDirectional_Late_T040_W070",
        "solver_type": "flowedit_bridge_directional",
        "pc_enable_below_t": 0.4,
        "pc_guidance_weight": 0.7,
    })
    exp.append(directional)

if RUN_CLIP_RECOVERY_CANDIDATE:
    recovery = dict(exp[-1])
    recovery.update({
        "exp_name": "FiveSample_BridgeDirMag_ClipRecovery_T050_W080_M025_R112",
        "solver_type": "flowedit_bridge_dir_mag",
        "pc_enable_below_t": 0.5,
        "pc_guidance_weight": 0.8,
        "pc_magnitude_weight": 0.25,
        "pc_magnitude_max": 1.12,
    })
    exp.append(recovery)

# If optional rows were added, write a temporary runtime YAML so the repo config stays clean.
RUNTIME_EXP_YAML = EXP_YAML
if len(exp) > 3:
    RUNTIME_EXP_YAML = f"outputs/{TAG}_runtime_exp.yaml"
    Path("outputs").mkdir(exist_ok=True)
    with open(RUNTIME_EXP_YAML, "w", encoding="utf-8") as f:
        yaml.safe_dump(exp, f, sort_keys=False)

exp_table = pd.DataFrame(exp)
display(exp_table[[
    "exp_name", "solver_type", "T_steps", "n_max", "tar_guidance_scale",
    "pc_guidance_lambda", "pc_guidance_gamma", "pc_enable_below_t",
    "pc_guidance_weight", "pc_magnitude_weight", "pc_magnitude_min", "pc_magnitude_max",
]].fillna(""))

sample_table = pd.DataFrame([
    {
        "case": Path(item["input_img"]).stem,
        "input_img": item["input_img"],
        "target": item["target_prompts"][0],
    }
    for item in dataset
])
display(sample_table)
print(f"Runtime YAML: {RUNTIME_EXP_YAML}")


In [ ]:
# 5) Run generation
import shutil
import subprocess
from pathlib import Path

cleanup_paths = [
    "outputs/run_summary.csv",
    QUALITY_PER_SAMPLE_CSV,
    QUALITY_SUMMARY_CSV,
    ARTIFACT_PER_SAMPLE_CSV,
    ARTIFACT_SUMMARY_CSV,
    COMPARISON_CSV,
    PER_SAMPLE_COMPARISON_CSV,
]
cleanup_paths.extend([f"outputs/{item['exp_name']}" for item in exp])
for path in cleanup_paths:
    p = Path(path)
    if p.is_dir():
        shutil.rmtree(p)
    elif p.exists():
        p.unlink()

Path(METRICS_DIR).mkdir(parents=True, exist_ok=True)
cmd = [
    "python", "run_script.py",
    "--device_number", "0",
    "--exp_yaml", RUNTIME_EXP_YAML,
    "--pipeline_load_mode", PIPELINE_LOAD_MODE,
]
print("Running:", " ".join(cmd))
subprocess.run(cmd, check=True)

run_summary = pd.read_csv("outputs/run_summary.csv")
display(run_summary[[
    "exp_name", "source_image", "solver_type", "estimated_nfe", "actual_nfe", "elapsed_seconds", "output_dir",
]])


In [ ]:
# 6) Evaluate CLIP/DINO and artifact proxy
import subprocess
import pandas as pd

subprocess.run([
    "python", "evaluate_clip_dino.py",
    "--run_summary_csv", "outputs/run_summary.csv",
    "--dataset_yaml", DATASET_YAML,
    "--out_samples", QUALITY_PER_SAMPLE_CSV,
    "--out_summary", QUALITY_SUMMARY_CSV,
], check=True)

subprocess.run([
    "python", "evaluate_artifact_proxy.py",
    "--run_summary_csv", "outputs/run_summary.csv",
    "--out_samples", ARTIFACT_PER_SAMPLE_CSV,
    "--out_summary", ARTIFACT_SUMMARY_CSV,
], check=True)

quality = pd.read_csv(QUALITY_SUMMARY_CSV)
artifact = pd.read_csv(ARTIFACT_SUMMARY_CSV)
common_keys = [
    "exp_name", "solver_type", "normalized_solver_type", "theory_family", "theory_formula",
    "midpoint_space", "correction_mode", "estimated_nfe", "actual_nfe",
    "pc_guidance_lambda", "pc_guidance_gamma", "pc_guidance_weight",
    "pc_magnitude_weight", "pc_magnitude_min", "pc_magnitude_max",
]
common_keys = [key for key in common_keys if key in quality.columns and key in artifact.columns]
summary = quality.merge(
    artifact[[
        *common_keys,
        "edited_artifact_proxy_mean", "artifact_proxy_delta_vs_source_mean",
        "edited_clipping_ratio_mean", "edited_high_saturation_ratio_mean",
    ]],
    on=common_keys,
    how="left",
)

for col in [
    "clip_alignment_mean", "dino_similarity_mean", "edit_preservation_score_mean",
    "edited_artifact_proxy_mean", "artifact_proxy_delta_vs_source_mean",
]:
    summary[col] = pd.to_numeric(summary[col], errors="coerce")

baseline_name = "FiveSample_OriginalFlowEdit_Euler18_Tar75"
baseline = summary.loc[summary["exp_name"] == baseline_name].iloc[0]
summary["delta_clip_vs_baseline"] = summary["clip_alignment_mean"] - baseline["clip_alignment_mean"]
summary["delta_dino_vs_baseline"] = summary["dino_similarity_mean"] - baseline["dino_similarity_mean"]
summary["delta_artifact_vs_baseline"] = summary["edited_artifact_proxy_mean"] - baseline["edited_artifact_proxy_mean"]
summary["clip_pp_vs_baseline"] = 100 * summary["delta_clip_vs_baseline"]
summary["dino_pp_vs_baseline"] = 100 * summary["delta_dino_vs_baseline"]
summary["candidate_readout"] = summary.apply(
    lambda r: "CLIP+DINO win" if r["delta_clip_vs_baseline"] > 0 and r["delta_dino_vs_baseline"] > 0 else "tradeoff / not both",
    axis=1,
)
summary.to_csv(COMPARISON_CSV, index=False)

display(summary[[
    "exp_name", "solver_type", "actual_nfe", "clip_alignment_mean", "dino_similarity_mean",
    "edited_artifact_proxy_mean", "delta_clip_vs_baseline", "delta_dino_vs_baseline",
    "clip_pp_vs_baseline", "dino_pp_vs_baseline", "candidate_readout",
]])
print(f"Wrote {COMPARISON_CSV}")


In [ ]:
# 7) Per-sample delta table
from pathlib import Path
import pandas as pd

quality_samples = pd.read_csv(QUALITY_PER_SAMPLE_CSV)
artifact_samples = pd.read_csv(ARTIFACT_PER_SAMPLE_CSV)
sample_keys = ["exp_name", "solver_type", "source_image", "target_index"]
samples = quality_samples.merge(
    artifact_samples[[
        *sample_keys,
        "edited_artifact_proxy", "artifact_proxy_delta_vs_source",
    ]],
    on=sample_keys,
    how="left",
)
samples["case"] = samples["source_image"].map(lambda p: Path(p).stem)
for col in ["clip_alignment", "dino_similarity", "edited_artifact_proxy"]:
    samples[col] = pd.to_numeric(samples[col], errors="coerce")

baseline_rows = samples[samples["exp_name"] == baseline_name][[
    "case", "clip_alignment", "dino_similarity", "edited_artifact_proxy",
]].rename(columns={
    "clip_alignment": "baseline_clip",
    "dino_similarity": "baseline_dino",
    "edited_artifact_proxy": "baseline_artifact",
})
per_sample = samples.merge(baseline_rows, on="case", how="left")
per_sample["delta_clip_vs_baseline"] = per_sample["clip_alignment"] - per_sample["baseline_clip"]
per_sample["delta_dino_vs_baseline"] = per_sample["dino_similarity"] - per_sample["baseline_dino"]
per_sample["delta_artifact_vs_baseline"] = per_sample["edited_artifact_proxy"] - per_sample["baseline_artifact"]
per_sample.to_csv(PER_SAMPLE_COMPARISON_CSV, index=False)

display(per_sample[[
    "case", "exp_name", "clip_alignment", "dino_similarity", "edited_artifact_proxy",
    "delta_clip_vs_baseline", "delta_dino_vs_baseline", "delta_artifact_vs_baseline",
]].sort_values(["case", "exp_name"]))
print(f"Wrote {PER_SAMPLE_COMPARISON_CSV}")


In [ ]:
# 8) Image comparison grid
import base64
import html
from pathlib import Path
from IPython.display import HTML, display
import pandas as pd
import yaml

samples = pd.read_csv(PER_SAMPLE_COMPARISON_CSV)
with open(DATASET_YAML, "r", encoding="utf-8") as f:
    dataset = yaml.safe_load(f)

exp_order = [item["exp_name"] for item in exp]
exp_labels = {
    "FiveSample_OriginalFlowEdit_Euler18_Tar75": "Euler18 baseline",
    "FiveSample_BridgeInterp_OldBest_L100_G200": "BridgeInterp old best",
    "FiveSample_BridgeDirMag_Balanced_T040_W070_M020_R110": "Dir-Mag balanced",
    "FiveSample_BridgeDirectional_Late_T040_W070": "Directional ref",
    "FiveSample_BridgeDirMag_ClipRecovery_T050_W080_M025_R112": "Dir-Mag clip recovery",
}

def img_tag(path, width=180):
    path = Path(path)
    if not path.exists():
        return f"<div style='width:{width}px;color:#b91c1c'>missing<br>{html.escape(str(path))}</div>"
    data = base64.b64encode(path.read_bytes()).decode("ascii")
    return f"<img src='data:image/png;base64,{data}' style='width:{width}px;border-radius:4px;border:1px solid #ddd'>"

parts = []
parts.append("<style>table.flowgrid{border-collapse:collapse;font-family:Arial,sans-serif;font-size:13px} .flowgrid th,.flowgrid td{border:1px solid #ddd;padding:8px;vertical-align:top} .metric{color:#374151;line-height:1.35}.prompt{max-width:260px;color:#4b5563}</style>")
parts.append("<table class='flowgrid'>")
parts.append("<tr><th>case</th><th>source</th>" + "".join(f"<th>{html.escape(exp_labels.get(e, e))}</th>" for e in exp_order) + "</tr>")

for item in dataset:
    case = Path(item["input_img"]).stem
    target_prompt = item["target_prompts"][0]
    row = [f"<tr><td><b>{html.escape(case)}</b><div class='prompt'>{html.escape(target_prompt)}</div></td>"]
    row.append(f"<td>{img_tag(item['input_img'])}</td>")
    for exp_name in exp_order:
        match = samples[(samples["case"] == case) & (samples["exp_name"] == exp_name)]
        if match.empty:
            row.append("<td>not run</td>")
            continue
        r = match.iloc[0]
        metric = (
            f"<div class='metric'>CLIP {r['clip_alignment']:.4f} ({r['delta_clip_vs_baseline']:+.4f})<br>"
            f"DINO {r['dino_similarity']:.4f} ({r['delta_dino_vs_baseline']:+.4f})<br>"
            f"Artifact {r['edited_artifact_proxy']:.4f} ({r['delta_artifact_vs_baseline']:+.4f})</div>"
        )
        row.append(f"<td>{img_tag(r['output_image'])}{metric}</td>")
    row.append("</tr>")
    parts.extend(row)

parts.append("</table>")
display(HTML("".join(parts)))


In [ ]:
# 9) Quick readout helper
import pandas as pd

summary = pd.read_csv(COMPARISON_CSV)
cols = [
    "exp_name", "clip_alignment_mean", "dino_similarity_mean",
    "delta_clip_vs_baseline", "delta_dino_vs_baseline",
    "edited_artifact_proxy_mean", "delta_artifact_vs_baseline",
    "candidate_readout",
]
display(summary[cols])

best_both = summary[(summary["delta_clip_vs_baseline"] > 0) & (summary["delta_dino_vs_baseline"] > 0)]
if len(best_both):
    print("At least one candidate beats baseline on both CLIP and DINO on this five-sample smoke test.")
else:
    print("No both-metric win yet. Use the per-sample table to see whether the issue is global or sample-specific.")
